### 국가별 법규 RAG 인덱스 생성

- 현재 확보한 실무형 PDF를 기준으로 문서를 불러온다.
- 문서를 청크로 분할하고 허깅페이스 임베딩으로 Redis 인덱스를 만든다.
- 흐름은 기존 RAG 실습과 같이 `문서 불러오기 -> 문서 분할하기 -> 임베딩 -> Vector DB 생성 및 저장 -> 검색기 생성 -> 프롬프트 생성 -> LLM 생성 -> 체인 실행` 순서를 따른다.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Redis
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache


In [ ]:
RAG_ROOT = Path.cwd()
DOCUMENT_ROOT = RAG_ROOT / "documents" / "trade_regulations"

PRACTICAL_KEYWORDS = [
    "form",
    "declaration",
    "import",
    "export",
    "clearance",
    "manual",
    "guide",
    "certificate",
    "customs",
]
NOISY_KEYWORDS = [
    "leaflet",
    "interim_results",
    "results_fy",
    "annual",
    "report",
    "strategy",
    "bulletin",
]


def is_practical_pdf(pdf_path: Path) -> bool:
    relative_parts = pdf_path.relative_to(DOCUMENT_ROOT).parts
    country_code = relative_parts[0] if len(relative_parts) >= 3 else ""
    agency_code = relative_parts[1] if len(relative_parts) >= 3 else ""
    lower_name = pdf_path.name.lower()

    if ".hwp.pdf" in lower_name:
        return False

    if any(keyword in lower_name for keyword in NOISY_KEYWORDS):
        return False

    if country_code == "KR" and agency_code == "CUSTOMS":
        return True

    if country_code == "US" and agency_code == "CBP":
        return any(keyword in lower_name for keyword in PRACTICAL_KEYWORDS)

    return False


FILE_PATHS = [
    str(pdf_path)
    for pdf_path in sorted(DOCUMENT_ROOT.rglob("*.pdf"))
    if is_practical_pdf(pdf_path)
]

print(f"실무형 PDF 수: {len(FILE_PATHS)}")
FILE_PATHS[:10]

In [ ]:
# 1단계, 문서 로드

all_docs = []

for file_path in FILE_PATHS:
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()

    pdf_path = Path(file_path)
    relative_parts = pdf_path.relative_to(DOCUMENT_ROOT).parts
    country_code = relative_parts[0] if len(relative_parts) >= 3 else ""
    agency_code = relative_parts[1] if len(relative_parts) >= 3 else ""

    for doc in docs:
        doc.metadata.update({
            "country_code": country_code,
            "agency_code": agency_code,
            "file_name": pdf_path.name,
            "local_path": file_path,
        })

    all_docs.extend(docs)

print(f"불러온 PDF 수: {len(FILE_PATHS)}")
print(f"문서의 페이지 수: {len(all_docs)}")

In [ ]:
print(all_docs[0].page_content)

In [ ]:
all_docs[0].__dict__

In [ ]:
# 2단계, 문서 분할

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(all_docs)
print(f"분할된 청크(조각)의 수: {len(split_documents)}")

In [ ]:
# 3단계, 임베딩

embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={'device': 'cpu'} 
)

In [ ]:
# Redis 시맨틱 캐시 설정

set_llm_cache(
    # RedisSemanticCache(
    #     redis_url="redis://localhost:6380",
    #     embedding=embeddings,
    #     score_threshold=0.1,
    # )
    None
)

In [ ]:
# 4단계, 벡터 DB

vectorstore = Redis.from_documents(
    documents=split_documents,
    embedding=embeddings,
    redis_url="redis://localhost:6380",
    index_name="trade_regulation_docs",
)

In [ ]:
vectorstore.similarity_search("What documents are required for importing into the United States?")[0].page_content

In [ ]:
# 5단계, 검색기 생성

retriever = vectorstore.as_retriever(
    search_type="mmr",
      search_kwargs={"k": 8, "fetch_k": 20})
retriever.invoke("entry documentation supporting information certificates of origin")

In [ ]:
# 6단계, 프롬프트 생성

prompt_template = PromptTemplate.from_template(
    """
      너는 국가별 수출입 법규와 통관 실무를 정리해주는 무역 브리핑 비서다.

      아래 문맥(Context)에 포함된 정보를 바탕으로, 질문과 관련된 내용을 최대한 폭넓고 실무적으로 정리하라.
      답변은 문맥에 나온 정보만 사용해야 하며, 문맥 밖의 사실을 추측해서 추가하지 마라.

      답변 원칙:
      1. 문맥에 직접 나온 서류, 증빙, 신고 항목, 주의사항을 최대한 빠짐없이 정리한다.
      2. 여러 청크에 흩어진 정보를 하나로 묶어서 정리한다.
      3. 기본적으로 필요한 문서와 특정 조건에서 추가로 필요한 문서를 구분한다.
      4. 문맥에 명시된 내용은 단정적으로 써도 되지만, 문맥상 가능성만 보이는 항목은 "추가 확인 필요"로 구분한다.
      5. 문맥에 일부 관련 정보만 있고 전체 목록은 아니어도, 확인 가능한 범위까지는 적극적으로 정리한다.
      6. "제공된 정보로는 확인이 불가능하다"는 문구는 정말 관련 정보가 거의 없을 때만 사용한다.
      7. 답변은 반드시 한국어로 작성한다.

      답변 형식:
      - 요약
      - 기본적으로 확인되는 서류/증빙
      - 조건에 따라 추가될 수 있는 서류/증빙
      - 주의사항
      - 추가 확인이 필요한 항목

      #Context:
      {context}

      #Question:
      {question}

      #Answer:
      """
)

In [ ]:
# 7단계, LLM 생성

llm = ChatOpenAI(
    model_name="gpt-5.4-nano", 
    temperature=0
)

In [ ]:
# 8단계, 체인 생성

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
question = "What documents are required for importing into the United States?"
answer = chain.invoke(question)
print(answer)